# GroupDNA — Your WhatsApp Group Chat Decoder
**Name:** Harshil Chauhan                                         
**Batch:** June 2024  


## Feature 1: Chat Parser

In [12]:
# i need to import datetime to work with times later
# and numpy for the grid thing
from datetime import datetime, timedelta
import numpy as np

# opening the file and reading everything inside it
fnme=input("File adderss/Name:")
with open(fnme, 'r') as f:
    content = f.read()

# splitting the big string into a list of lines
lines = content.split('\n')

# this list will hold all the msgs i find
msgs = []

# keeping count of the weird lines i want to skip
sys_cnt = 0
med_cnt = 0
del_cnt = 0

# tracking how many media and deleted msgs each person sent
med_pp = {}
del_pp = {}

# i need this to handle msgs that span multiple lines
prev_idx = -1

for line in lines:
    line = line.strip()

    # skip blank lines
    if line == '':
        continue

    # if the line is too short it cant be a real message
    if len(line) < 8:
        if prev_idx >= 0:
            msgs[prev_idx]['text'] = msgs[prev_idx]['text'] + ' ' + line
        continue

    # checking if the line starts with a date like 01/04/24
    head8 = line[:8]
    is_date = head8[2] == '/' and head8[5] == '/'

    # if it doesnt start with a date its a continuation of the previous message
    if not is_date:
        if prev_idx >= 0:
            msgs[prev_idx]['text'] = msgs[prev_idx]['text'] + ' ' + line
        continue

    # every real line should have ' - ' to separate timestamp from the rest
    if ' - ' not in line:
        continue

    # splitting into timestamp and everything after the dash
    parts = line.split(' - ', 1)
    ts = parts[0].strip()
    rest = parts[1].strip()

    # system msgs dont have a colon after the sender name
    if ': ' not in rest:
        sys_cnt = sys_cnt + 1
        continue

    # now splitting sender name from the actual message text
    parts2 = rest.split(': ', 1)
    sender = parts2[0].strip()
    text = parts2[1].strip()

    # checking if it was a media file that got omitted
    if text == '<Media omitted>':
        if sender not in med_pp:
            med_pp[sender] = 0
        med_pp[sender] = med_pp[sender] + 1
        med_cnt = med_cnt + 1
        msg = {'timestamp': ts, 'sender': sender, 'text': text, 'is_media': True, 'is_deleted': False}
        msgs.append(msg)
        prev_idx = len(msgs) - 1
        continue

    # checking if the person deleted this message
    if text == 'This message was deleted':
        if sender not in del_pp:
            del_pp[sender] = 0
        del_pp[sender] = del_pp[sender] + 1
        del_cnt = del_cnt + 1
        msg = {'timestamp': ts, 'sender': sender, 'text': text, 'is_media': False, 'is_deleted': True}
        msgs.append(msg)
        prev_idx = len(msgs) - 1
        continue

    # this is a normal real message so i store it
    msg = {'timestamp': ts, 'sender': sender, 'text': text, 'is_media': False, 'is_deleted': False}
    msgs.append(msg)
    prev_idx = len(msgs) - 1

# keeping only the real msgs for analysis, skipping media and deleted
real_msgs = []
for m in msgs:
    if not m['is_media'] and not m['is_deleted']:
        real_msgs.append(m)

# finding out who all the senders are
senders = []
for m in real_msgs:
    if m['sender'] not in senders:
        senders.append(m['sender'])

# looking through the system msgs to find the group name
grp_name = 'Unknown Group'
for line in lines:
    line = line.strip()
    if 'created group' in line and '"' in line:
        start = line.index('"') + 1
        end = line.index('"', start)
        grp_name = line[start:end]
        break

print('Successfully parsed', len(real_msgs), 'msgs from', len(senders), 'participants')
print('Skipped', sys_cnt, 'system msgs,', med_cnt, 'media-omitted,', del_cnt, 'deleted msgs')
#I had a hard time understanding how this worked, and took help from various sources including ai to do this part.

File adderss/Name:hostel_bois.txt
Successfully parsed 3127 messages from 6 participants
Skipped 4 system messages, 32 media-omitted, 15 deleted messages


## Feature 2: Group Overview

In [13]:
# counting how many msgs each person sent
cnt_pp = {}
for m in real_msgs:
    sender = m['sender']
    if sender not in cnt_pp:
        cnt_pp[sender] = 0
    cnt_pp[sender] = cnt_pp[sender] + 1

# sorting by message count so the most active person shows up first
ranked = sorted(cnt_pp, key=lambda x: cnt_pp[x], reverse=True)

# converting all timestamps to datetime objects so i can find the first and last date
dates = []
for m in real_msgs:
    dt = datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M')
    dates.append(dt)

start_dt = dates[0]
end_dt = dates[-1]
num_days = (end_dt - start_dt).days + 1
tot_msgs = len(real_msgs)

print('=' * 60)
print('GROUP OVERVIEW')
print('=' * 60)
print('Group            : ' + grp_name)
print('Period           : ' + start_dt.strftime('%d %B %Y') + ' to ' + end_dt.strftime('%d %B %Y') + ' (' + str(num_days) + ' days)')
print('Total msgs   : ' + str(tot_msgs))
print('Participants     : ' + str(len(ranked)))
print('System msgs  : ' + str(sys_cnt))
print('Media shared     : ' + str(med_cnt))
print('Deleted msgs : ' + str(del_cnt))
print()

# printing each persons message count with a little bar chart
print('MESSAGES PER PERSON')
print('-' * 60)
for sender in ranked:
    count = cnt_pp[sender]
    pct = (count / tot_msgs) * 100
    # bar length is based on percentage so it looks proportional
    bar = chr(9608) * int(pct / 2)
    print(sender.ljust(10) + ' : ' + str(count).rjust(4) + ' (' + str(round(pct, 1)).rjust(5) + '%)  ' + bar)


GROUP OVERVIEW
Group            : Hostel Bois 4ever
Period           : 01 April 2024 to 30 May 2024 (60 days)
Total messages   : 3127
Participants     : 6
System messages  : 4
Media shared     : 32
Deleted messages : 15

MESSAGES PER PERSON
------------------------------------------------------------
Rahul      :  940 ( 30.1%)  ███████████████
Priya      :  712 ( 22.8%)  ███████████
Neha       :  624 ( 20.0%)  █████████
Aman       :  484 ( 15.5%)  ███████
Karan      :  345 ( 11.0%)  █████
Vikas      :   22 (  0.7%)  


## Feature 3: Most Active Day and Hour

In [14]:
# making two dicts, one for daily counts and one for hourly counts
day_cnt = {}
hr_cnt = {}

for m in real_msgs:
    dt = datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M')
    dkey = dt.strftime('%d %B %Y')
    hkey = dt.hour

    if dkey not in day_cnt:
        day_cnt[dkey] = 0
    day_cnt[dkey] = day_cnt[dkey] + 1

    if hkey not in hr_cnt:
        hr_cnt[hkey] = 0
    hr_cnt[hkey] = hr_cnt[hkey] + 1

# using max with a key to find whichever day had the most msgs
top_day = max(day_cnt, key=lambda x: day_cnt[x])
top_day_cnt = day_cnt[top_day]

top_hr = max(hr_cnt, key=lambda x: hr_cnt[x])
top_hr_cnt = hr_cnt[top_hr]
hr_avg = top_hr_cnt / num_days

print('=' * 60)
print('MOST ACTIVE TIMES')
print('=' * 60)
print('Busiest day   : ' + top_day + ' (' + str(top_day_cnt) + ' msgs)')
print('Busiest hour  : ' + str(top_hr).zfill(2) + ':00 - ' + str(top_hr + 1).zfill(2) + ':00 (avg ' + str(round(hr_avg, 1)) + ' msgs per day)')
print()

# showing all 24 hours sorted so i can see which hours are bussiest
print('MESSAGES BY HOUR (all hours)')
print('-' * 60)
max_hr = hr_cnt[top_hr]
for h in range(24):
    count = 0
    if h in hr_cnt:
        count = hr_cnt[h]
    blen = int((count / max_hr) * 25)
    bar = chr(9608) * blen
    print(str(h).zfill(2) + ':00  ' + bar.ljust(27) + ' ' + str(count))

MOST ACTIVE TIMES
Busiest day   : 04 May 2024 (74 messages)
Busiest hour  : 18:00 - 19:00 (avg 4.1 messages per day)

MESSAGES BY HOUR (all hours)
------------------------------------------------------------
00:00  █████                       56
01:00  ████████                    82
02:00  ████████                    83
03:00  ███████                     77
04:00  ███████████                 109
05:00  ██                          28
06:00  ███                         33
07:00  █████                       55
08:00  ███████████                 117
09:00  ███████████████             149
10:00  ████████████████            158
11:00  ███████████                 113
12:00  ███████████████████         190
13:00  ███████████████             156
14:00  ████████████████            161
15:00  █████████████               128
16:00  ██████████████████          185
17:00  █████████████████           170
18:00  █████████████████████████   244
19:00  ██████████████████████      224
20:00  ████████████

## Feature 4: Activity Heatmap (NumPy)

In [15]:
# mapping each sender name to a row number in the matrix
p_idx = {}
for i in range(len(ranked)):
    p_idx[ranked[i]] = i

# creating a 6x24 matrix filled with zeros using numpy
# rows are people, columns are hours of the day
grid = np.zeros((len(ranked), 24), dtype=int)

# going through every message and adding 1 to the right cell
for m in real_msgs:
    sender = m['sender']
    dt = datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M')
    hour = dt.hour
    row = p_idx[sender]
    grid[row][hour] = grid[row][hour] + 1

print('=' * 60)
print('ACTIVITY HEATMAP (msgs by hour, 3-hour blocks)')
print('=' * 60)

# printing the hour labels across the top
hr_hdr = '          '
for h in range(0, 24, 3):
    hr_hdr = hr_hdr + str(h).zfill(2) + '  '
print(hr_hdr)

# printing each persons row using block characters to show activity levels
# . means almost nothing, full block means very active in that time slot
for i in range(len(ranked)):
    sender = ranked[i]
    row = grid[i]
    max_val = row.max()
    row_str = sender.ljust(10)
    for h in range(0, 24, 3):
        csum = 0
        for hh in range(h, min(h + 3, 24)):
            csum = csum + row[hh]
        if max_val == 0:
            block = '.'
        else:
            ratio = csum / (max_val * 3)
            if ratio == 0:
                block = '.'
            elif ratio < 0.25:
                block = chr(9617)
            elif ratio < 0.5:
                block = chr(9618)
            else:
                block = chr(9619)
        row_str = row_str + block + '   '
    print(row_str)

print()
# also printing the row totals using numpy sum
print('TOTAL MESSAGES PER PERSON (from numpy matrix)')
print('-' * 60)
row_tots = grid.sum(axis=1)
max_tot = row_tots.max()
for i in range(len(ranked)):
    sender = ranked[i]
    total = row_tots[i]
    blen = int((total / max_tot) * 25)
    bar = chr(9608) * blen
    print(sender.ljust(10) + ' : ' + str(total).rjust(4) + '  ' + bar)
# This part was also something i was having issues with and took help from various sources including AI

ACTIVITY HEATMAP (messages by hour, 3-hour blocks)
          00  03  06  09  12  15  18  21  
Rahul     ░   ░   ░   ░   ▒   ▓   ▓   ▓   
Priya     .   .   ▒   ▓   ▓   ▓   ▓   ▒   
Neha      .   ░   ▒   ▓   ▓   ▒   ▓   ▒   
Aman      ▓   ▓   .   .   ░   ░   ░   ▒   
Karan     .   .   ░   ▒   ▓   ▓   ▓   ▒   
Vikas     .   .   ▒   ░   ▒   ▓   ▓   ▒   

TOTAL MESSAGES PER PERSON (from numpy matrix)
------------------------------------------------------------
Rahul      :  940  █████████████████████████
Priya      :  712  ██████████████████
Neha       :  624  ████████████████
Aman       :  484  ████████████
Karan      :  345  █████████
Vikas      :   22  


## Feature 5: Top Words

In [16]:
# these are really common words that dont tell us anything interesting
skip_words = ['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
              'it', 'me', 'my', 'you', 'he', 'she', 'we', 'they', 'this', 'that',
              'but', 'are', 'was', 'be', 'been', 'have', 'has', 'do', 'did', 'not',
              'with', 'so', 'at', 'by', 'from', 'up', 'about', 'into', 'then',
              'its', 'also', 'as', 'an', 'if', 'no', 'just', 'will', 'get', 'got',
              'can', 'all', 'out', 'what', 'your', 'his', 'her', 'their', 'our']

# this dict will store how many times each word appears
wrd_cnt = {}

for m in real_msgs:
    # lowercase so Bhai and bhai count as the same word
    text = m['text'].lower()
    raw_wds = text.split()
    for word in raw_wds:
        # removing punctuation from the start and end of the word
        cleaned = word
        while len(cleaned) > 0 and not cleaned[0].isalpha():
            cleaned = cleaned[1:]
        while len(cleaned) > 0 and not cleaned[-1].isalpha():
            cleaned = cleaned[:-1]
        if cleaned == '':
            continue
        if cleaned in skip_words:
            continue
        # skipping single characters like 'k' or 'n'
        if len(cleaned) < 2:
            continue
        if cleaned not in wrd_cnt:
            wrd_cnt[cleaned] = 0
        wrd_cnt[cleaned] = wrd_cnt[cleaned] + 1

# sorting all words by count and taking the top 10
wrd_ranked = sorted(wrd_cnt, key=lambda x: wrd_cnt[x], reverse=True)
top_wds = wrd_ranked[:10]
max_wrd = wrd_cnt[top_wds[0]]

print('=' * 60)
print("THIS GROUP'S FAVOURITE WORDS")
print('=' * 60)
# the bar length is proportional to how often the word appears
for word in top_wds:
    count = wrd_cnt[word]
    blen = int((count / max_wrd) * 20)
    bar = chr(9608) * blen
    print(word.ljust(14) + ' ' + bar.ljust(22) + ' ' + str(count))

print()
# also showing per person what their most used word is
print('TOP WORD PER PERSON')
print('-' * 60)
for sender in ranked:
    p_wrd_cnt = {}
    for m in real_msgs:
        if m['sender'] == sender:
            text = m['text'].lower()
            raw_wds = text.split()
            for word in raw_wds:
                cleaned = word
                while len(cleaned) > 0 and not cleaned[0].isalpha():
                    cleaned = cleaned[1:]
                while len(cleaned) > 0 and not cleaned[-1].isalpha():
                    cleaned = cleaned[:-1]
                if cleaned == '' or cleaned in skip_words or len(cleaned) < 2:
                    continue
                if cleaned not in p_wrd_cnt:
                    p_wrd_cnt[cleaned] = 0
                p_wrd_cnt[cleaned] = p_wrd_cnt[cleaned] + 1
    if len(p_wrd_cnt) > 0:
        best_wrd = max(p_wrd_cnt, key=lambda x: p_wrd_cnt[x])
        print(sender.ljust(10) + ' : ' + best_wrd + ' (' + str(p_wrd_cnt[best_wrd]) + ' times)')


THIS GROUP'S FAVOURITE WORDS
how            ████████████████████   321
guys           ███████████████████    318
hai            ████████████████       268
am             ████████████████       260
today          ████████████████       257
which          ████████████           202
everyone       ███████████            187
telling        ███████████            179
bhai           █████████              160
one            █████████              157

TOP WORD PER PERSON
------------------------------------------------------------
Rahul      : hai (263 times)
Priya      : please (141 times)
Neha       : am (148 times)
Aman       : am (98 times)
Karan      : how (295 times)
Vikas      : hai (5 times)


## Feature 6: Response Speed and Silent Streaks

In [17]:
# storing the time gaps (in seconds) for each person when they replied
rep_gaps = {}
for sender in ranked:
    rep_gaps[sender] = []

# going through every message and checking how long it took to reply
for i in range(1, len(real_msgs)):
    cur_msg = real_msgs[i]
    prev_msg = real_msgs[i - 1]

    # not counting it as a reply if the same person sent both msgs
    if cur_msg['sender'] == prev_msg['sender']:
        continue

    cur_t = datetime.strptime(cur_msg['timestamp'], '%d/%m/%y, %H:%M')
    prev_t = datetime.strptime(prev_msg['timestamp'], '%d/%m/%y, %H:%M')

    gap_sec = (cur_t - prev_t).total_seconds()

    # ignoring negative gaps and overnight gaps
    if gap_sec < 0:
        continue
    if gap_sec > 86400:
        continue

    rep_gaps[cur_msg['sender']].append(gap_sec)

# calculating average response time for each person
avg_rep = {}
for sender in ranked:
    gaps = rep_gaps[sender]
    if len(gaps) == 0:
        avg_rep[sender] = 999999
    else:
        tot_gap = 0
        for g in gaps:
            tot_gap = tot_gap + g
        avg_rep[sender] = tot_gap / len(gaps)

# finding who replies fastest and slowest
fastest = ranked[0]
for sender in ranked:
    if avg_rep[sender] < avg_rep[fastest]:
        fastest = sender

slowest = ranked[0]
for sender in ranked:
    if avg_rep[sender] > avg_rep[slowest]:
        slowest = sender

# helper function to make seconds look nice
def seconds_to_readable(secs):
    if secs < 60:
        return str(round(secs)) + ' seconds'
    elif secs < 3600:
        return str(round(secs / 60, 1)) + ' minutes'
    else:
        return str(round(secs / 3600, 1)) + ' hours'

# finding which days each person was active
active_days = {}
for sender in ranked:
    active_days[sender] = []

for m in real_msgs:
    dt = datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M')
    dstr = dt.strftime('%Y-%m-%d')
    if dstr not in active_days[m['sender']]:
        active_days[m['sender']].append(dstr)

# finding the longest silent streak for each person
# a streak is consecutive days with zero msgs
lng_streak = {}
for sender in ranked:
    act_set = active_days[sender]
    all_days = []
    cur_day = start_dt
    while cur_day <= end_dt:
        all_days.append(cur_day.strftime('%Y-%m-%d'))
        cur_day = cur_day + timedelta(days=1)
    max_str = 0
    cur_str = 0
    for day in all_days:
        if day not in act_set:
            cur_str = cur_str + 1
            if cur_str > max_str:
                max_str = cur_str
        else:
            cur_str = 0
    lng_streak[sender] = max_str

streak_rank = sorted(ranked, key=lambda x: lng_streak[x], reverse=True)

print('=' * 60)
print('RESPONSE PATTERNS')
print('=' * 60)
print('Fastest replier : ' + fastest + ' (' + seconds_to_readable(avg_rep[fastest]) + ')')
print('Slowest replier : ' + slowest + ' (' + seconds_to_readable(avg_rep[slowest]) + ')')
print()

# showing everyone sorted by how fast they reply on average
print('AVERAGE RESPONSE TIME PER PERSON')
print('-' * 60)
speed_rank = sorted(ranked, key=lambda x: avg_rep[x])
max_rep = avg_rep[slowest]
for sender in speed_rank:
    avg = avg_rep[sender]
    blen = int(((max_rep - avg) / max_rep) * 25)
    bar = chr(9608) * blen
    print(sender.ljust(10) + ' : ' + seconds_to_readable(avg).ljust(15) + '  ' + bar)
print()

print('LONGEST SILENT STREAKS (consecutive days with zero msgs)')
print('-' * 60)
mstr = lng_streak[streak_rank[0]]
for sender in streak_rank:
    streak = lng_streak[sender]
    if streak == 0:
        print(sender.ljust(10) + ' : 0 days (never went silent)')
    else:
        blen = int((streak / mstr) * 25) if mstr > 0 else 0
        bar = chr(9608) * blen
        print(sender.ljust(10) + ' : ' + str(streak).rjust(2) + ' days  ' + bar)


RESPONSE PATTERNS
Fastest replier : Vikas (34.9 minutes)
Slowest replier : Aman (54.9 minutes)

AVERAGE RESPONSE TIME PER PERSON
------------------------------------------------------------
Vikas      : 34.9 minutes     █████████
Rahul      : 35.2 minutes     ████████
Karan      : 36.8 minutes     ████████
Neha       : 41.3 minutes     ██████
Priya      : 42.6 minutes     █████
Aman       : 54.9 minutes     

LONGEST SILENT STREAKS (consecutive days with zero messages)
------------------------------------------------------------
Vikas      : 11 days  █████████████████████████
Rahul      : 0 days (never went silent)
Priya      : 0 days (never went silent)
Neha       : 0 days (never went silent)
Aman       : 0 days (never went silent)
Karan      : 0 days (never went silent)


## Feature 7: Personality Archetype Detection

In [18]:
# each function below checks one archetype and returns a score
# the higher the score the more that person fits that archetype

def get_spammer_score(sender, all_messages):
    # counting how many msgs in a row this person tends to send
    bursts = []
    cur_burst = 1
    for i in range(1, len(all_messages)):
        if all_messages[i]['sender'] == sender and all_messages[i-1]['sender'] == sender:
            cur_burst = cur_burst + 1
        elif all_messages[i-1]['sender'] == sender:
            bursts.append(cur_burst)
            cur_burst = 1
    if len(bursts) == 0:
        return 0
    total = 0
    for b in bursts:
        total = total + b
    return total / len(bursts)

def get_group_mom_score(sender, all_messages):
    # counting how many times this person uses caring words
    care_wds = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',
                    'reminder', 'drink water', "don't forget", 'doing okay', 'feel better']
    count = 0
    for m in all_messages:
        if m['sender'] == sender:
            text = m['text'].lower()
            for word in care_wds:
                if word in text:
                    count = count + 1
    return count

def get_night_owl_score(sender, mat, pidx):
    # checking what fraction of their msgs were sent between 11pm and 5am
    row = mat[pidx]
    night_hrs = list(range(23, 24)) + list(range(0, 5))
    night_cnt = 0
    for h in night_hrs:
        night_cnt = night_cnt + row[h]
    total = row.sum()
    if total == 0:
        return 0
    return night_cnt / total

def get_storyteller_score(sender, all_messages):
    # average number of words per message, longer msgs = storyteller
    tot_wds = 0
    mcnt = 0
    for m in all_messages:
        if m['sender'] == sender:
            tot_wds = tot_wds + len(m['text'].split())
            mcnt = mcnt + 1
    if mcnt == 0:
        return 0
    return tot_wds / mcnt

def get_drama_queen_score(sender, all_messages):
    # checking how many of their msgs are all caps or have 2+ exclamation marks
    caps_cnt = 0
    eligible = 0
    for m in all_messages:
        if m['sender'] == sender:
            text = m['text']
            if len(text) < 3:
                continue
            eligible = eligible + 1
            alpha = ''
            for c in text:
                if c.isalpha():
                    alpha = alpha + c
            if len(alpha) > 0 and alpha.isupper():
                caps_cnt = caps_cnt + 1
            elif text.count('!') >= 2:
                caps_cnt = caps_cnt + 1
    if eligible == 0:
        return 0
    return caps_cnt / eligible

def get_ghost_score(sender, act_dict, ndays):
    # fraction of days they were completely silent
    act_cnt = len(act_dict[sender])
    sil_cnt = ndays - act_cnt
    return sil_cnt / ndays

def get_comedian_score(sender, all_messages):
    # what fraction of their msgs have laugh words in them
    laugh_wds = ['lol', 'lmao', 'haha', 'rofl', 'lmfao', 'hahaha', 'hehe']
    count = 0
    total = 0
    for m in all_messages:
        if m['sender'] == sender:
            total = total + 1
            text = m['text'].lower()
            for w in laugh_wds:
                if w in text:
                    count = count + 1
                    break
    if total == 0:
        return 0
    return count / total

def get_question_master_score(sender, all_messages):
    # how many of their msgs end with a question mark
    q_count = 0
    total = 0
    for m in all_messages:
        if m['sender'] == sender:
            total = total + 1
            if m['text'].strip().endswith('?'):
                q_count = q_count + 1
    if total == 0:
        return 0
    return q_count / total

# computing scores for every person for every archetype
arch_scores = {}
for sender in ranked:
    idx = p_idx[sender]
    scores = {}
    # dividing by big numbers so all scores are between 0 and 1 roughly
    scores['THE SPAMMER'] = get_spammer_score(sender, real_msgs) / 10.0
    scores['THE GROUP MOM'] = get_group_mom_score(sender, real_msgs) / 300.0
    scores['THE NIGHT OWL'] = get_night_owl_score(sender, grid, idx)
    scores['THE STORYTELLER'] = get_storyteller_score(sender, real_msgs) / 100.0
    scores['THE DRAMA QUEEN'] = get_drama_queen_score(sender, real_msgs)
    scores['THE GHOST'] = get_ghost_score(sender, active_days, num_days)
    scores['THE COMEDIAN'] = get_comedian_score(sender, real_msgs)
    scores['THE QUESTION MASTER'] = get_question_master_score(sender, real_msgs)
    arch_scores[sender] = scores

# assigning whoever has the highest score in each archetype to that archetype
archetypes = {}
for sender in ranked:
    scores = arch_scores[sender]
    best_arch = None
    best_sc = -1
    for archetype in scores:
        if scores[archetype] > best_sc:
            best_sc = scores[archetype]
            best_arch = archetype
    archetypes[sender] = best_arch

# making a short description line for each person based on their archetype
arch_info = {}
for sender in ranked:
    idx = p_idx[sender]
    archetype = archetypes[sender]
    if archetype == 'THE SPAMMER':
        detail = 'avg ' + str(round(get_spammer_score(sender, real_msgs), 1)) + ' msgs in a row'
    elif archetype == 'THE GROUP MOM':
        detail = 'caring keyword score: ' + str(get_group_mom_score(sender, real_msgs))
    elif archetype == 'THE NIGHT OWL':
        pct = round(get_night_owl_score(sender, grid, idx) * 100, 1)
        detail = str(pct) + '% msgs between 23h-04h'
    elif archetype == 'THE STORYTELLER':
        detail = 'avg ' + str(round(get_storyteller_score(sender, real_msgs), 1)) + ' words per msg'
    elif archetype == 'THE DRAMA QUEEN':
        pct = round(get_drama_queen_score(sender, real_msgs) * 100, 1)
        detail = str(pct) + '% ALL-CAPS msgs'
    elif archetype == 'THE GHOST':
        pct = round(get_ghost_score(sender, active_days, num_days) * 100, 1)
        detail = 'silent ' + str(pct) + '% of days'
    elif archetype == 'THE COMEDIAN':
        pct = round(get_comedian_score(sender, real_msgs) * 100, 1)
        detail = str(pct) + '% msgs with laugh words'
    elif archetype == 'THE QUESTION MASTER':
        pct = round(get_question_master_score(sender, real_msgs) * 100, 1)
        detail = str(pct) + '% msgs end with ?'
    else:
        detail = ''
    arch_info[sender] = detail

print('=' * 60)
print('PERSONALITY ARCHETYPES')
print('=' * 60)
for sender in ranked:
    archetype = archetypes[sender]
    detail = arch_info[sender]
    print(sender.ljust(10) + ' -> ' + archetype.ljust(22) + ' (' + detail + ')')

print()
# also showing the full score breakdown so i can see how close people were
print('FULL SCORE BREAKDOWN')
print('-' * 60)
for sender in ranked:
    print(sender + ':')
    scores = arch_scores[sender]
    arch_ranked = sorted(scores, key=lambda x: scores[x], reverse=True)
    for archetype in arch_ranked:
        score = scores[archetype]
        blen = int(score * 25)
        bar = chr(9608) * blen
        marker = ' <--' if archetype == archetypes[sender] else ''
        print('  ' + archetype.ljust(22) + ' ' + bar.ljust(27) + ' ' + str(round(score, 3)) + marker)
    print()
# This was the Longest Sequence and the hardest to code, after being stuck for 1 day and half i was able to do it

PERSONALITY ARCHETYPES
Rahul      -> THE SPAMMER            (avg 4.5 msgs in a row)
Priya      -> THE GROUP MOM          (caring keyword score: 639)
Neha       -> THE DRAMA QUEEN        (63.3% ALL-CAPS messages)
Aman       -> THE NIGHT OWL          (80.4% msgs between 23h-04h)
Karan      -> THE STORYTELLER        (avg 57.0 words per msg)
Vikas      -> THE GHOST              (silent 73.3% of days)

FULL SCORE BREAKDOWN
------------------------------------------------------------
Rahul:
  THE SPAMMER            ███████████                 0.448 <--
  THE NIGHT OWL          ███                         0.135
  THE QUESTION MASTER                                0.038
  THE COMEDIAN                                       0.032
  THE STORYTELLER                                    0.026
  THE GROUP MOM                                      0.0
  THE DRAMA QUEEN                                    0.0
  THE GHOST                                          0.0

Priya:
  THE GROUP MOM          ███████

## Feature 8: Final Report

In [19]:
# this is the big final report that puts everything together nicely
print()
print('=' * 60)
print('  GroupDNA -- YOUR WHATSAPP GROUP, DECODER')
print('=' * 60)
print()

print('GROUP OVERVIEW')
print('-' * 60)
print('Group            :   ' + grp_name)
print('Period           :   ' + start_dt.strftime('%d %B %Y') + ' to ' + end_dt.strftime('%d %B %Y') + ' (' + str(num_days) + ' days)')
print('Total msgs   :   ' + str(tot_msgs))
print('Participants     :   ' + str(len(ranked)))
print('System msgs  :   ' + str(sys_cnt))
print('Media shared     :   ' + str(med_cnt))
print('Deleted msgs :   ' + str(del_cnt))
print()

# msgs per person with bar chart
print('MESSAGES PER PERSON')
print('-' * 60)
for sender in ranked:
    count = cnt_pp[sender]
    pct = (count / tot_msgs) * 100
    bar = chr(9608) * int(pct / 2)
    print(sender.ljust(10) + ' : ' + str(count).rjust(4) + ' (' + str(round(pct, 1)).rjust(5) + '%)  ' + bar)
print()

# busiest times
print('MOST ACTIVE TIMES')
print('-' * 60)
print('Busiest day   : ' + top_day + ' (' + str(top_day_cnt) + ' msgs)')
print('Busiest hour  : ' + str(top_hr).zfill(2) + ':00 - ' + str(top_hr + 1).zfill(2) + ':00 (avg ' + str(round(hr_avg, 1)) + ' per day)')
print()

# the grid
print('ACTIVITY HEATMAP (msgs by hour, 3-hour blocks)')
print('-' * 60)
hr_hdr = '          '
for h in range(0, 24, 3):
    hr_hdr = hr_hdr + str(h).zfill(2) + '  '
print(hr_hdr)
for i in range(len(ranked)):
    sender = ranked[i]
    row = grid[i]
    max_val = row.max()
    row_str = sender.ljust(10)
    for h in range(0, 24, 3):
        csum = 0
        for hh in range(h, min(h + 3, 24)):
            csum = csum + row[hh]
        if max_val == 0:
            block = '.'
        else:
            ratio = csum / (max_val * 3)
            if ratio == 0:
                block = '.'
            elif ratio < 0.25:
                block = chr(9617)
            elif ratio < 0.5:
                block = chr(9618)
            else:
                block = chr(9619)
        row_str = row_str + block + '   '
    print(row_str)
print()

# top words with bars
print("THIS GROUP'S FAVOURITE WORDS")
print('-' * 60)
for word in top_wds:
    count = wrd_cnt[word]
    blen = int((count / max_wrd) * 20)
    bar = chr(9608) * blen
    print(word.ljust(14) + ' ' + bar.ljust(22) + ' ' + str(count))
print()

# response speed with bars
print('RESPONSE PATTERNS')
print('-' * 60)
print('Fastest replier : ' + fastest + ' (' + seconds_to_readable(avg_rep[fastest]) + ')')
print('Slowest replier : ' + slowest + ' (' + seconds_to_readable(avg_rep[slowest]) + ')')
print()

# silent streaks with bars
print('LONGEST SILENT STREAKS')
print('-' * 60)
mstr = lng_streak[streak_rank[0]]
for sender in streak_rank:
    streak = lng_streak[sender]
    if streak == 0:
        print(sender.ljust(10) + ' : 0 days (never went silent)')
    else:
        blen = int((streak / mstr) * 25) if mstr > 0 else 0
        bar = chr(9608) * blen
        print(sender.ljust(10) + ' : ' + str(streak).rjust(2) + ' days  ' + bar)
print()

# the archetypes
print('PERSONALITY ARCHETYPES')
print('-' * 60)
for sender in ranked:
    archetype = archetypes[sender]
    detail = arch_info[sender]
    print(sender.ljust(10) + ' -> ' + archetype.ljust(22) + ' (' + detail + ')')
print()

print('=' * 60)
print('Generated by GroupDNA  |  Built with Python + NumPy')
print('=' * 60)



  GroupDNA -- YOUR WHATSAPP GROUP, DECODER

GROUP OVERVIEW
------------------------------------------------------------
Group            : Hostel Bois 4ever
Period           : 01 April 2024 to 30 May 2024 (60 days)
Total messages   : 3127
Participants     : 6
System messages  : 4
Media shared     : 32
Deleted messages : 15

MESSAGES PER PERSON
------------------------------------------------------------
Rahul      :  940 ( 30.1%)  ███████████████
Priya      :  712 ( 22.8%)  ███████████
Neha       :  624 ( 20.0%)  █████████
Aman       :  484 ( 15.5%)  ███████
Karan      :  345 ( 11.0%)  █████
Vikas      :   22 (  0.7%)  

MOST ACTIVE TIMES
------------------------------------------------------------
Busiest day   : 04 May 2024 (74 messages)
Busiest hour  : 18:00 - 19:00 (avg 4.1 per day)

ACTIVITY HEATMAP (messages by hour, 3-hour blocks)
------------------------------------------------------------
          00  03  06  09  12  15  18  21  
Rahul     ░   ░   ░   ░   ▒   ▓   ▓   ▓   
Pr

## Reflection

**Hardest part:** Writing the parser that handles all the edge cases — system messages, media omitted, deleted messages, and multi-line messages took the most debugging.

**What I would do differently:** I would probably build the parser first as a completely separate function and test it thoroughly before moving on to any analysis feature.

**My own archetype:** I didnt have huge group chats to work with but the ones i did, seemed to work just fine

**Constraints followed:**
- No pandas
- No matplotlib
- No collections.Counter
- No regex
- Only Python fundamentals + NumPy + datetime
